In [1]:
import pandas as pd
from pathlib import Path

print("Cargando archivos...")

# ========= RUTAS =========

RAW_PATH = r"C:\Users\34642\Desktop\FLOWMAP ANALYTICS\segundo proyecto\POWER BI\data\raw"

OUTPUT_PATH = r"C:\Users\34642\Desktop\FLOWMAP ANALYTICS\segundo proyecto\POWER BI\data_processed"

# ========= TRAIN PARQUET =========

train = pd.read_parquet(
    rf"{OUTPUT_PATH}\train.parquet"
)

# ========= MAESTROS =========

items = pd.read_csv(
    rf"{RAW_PATH}\items.csv"
)

stores = pd.read_csv(
    rf"{RAW_PATH}\stores.csv"
)

print("Train:", train.shape)
print("Items:", items.shape)
print("Stores:", stores.shape)

Path(OUTPUT_PATH).mkdir(
    exist_ok=True,
    parents=True
)

print("Carga completada.")

Cargando archivos...
Train: (125497040, 6)
Items: (4100, 4)
Stores: (54, 5)
Carga completada.


In [2]:
#SALES_STORE_DAY.PARQUET
print("Creando sales_store_day...")

sales_store_day = (
    train
    .groupby(
        ["date", "store_nbr"],
        as_index=False
    )
    ["unit_sales"]
    .sum()
)

sales_store_day.rename(
    columns={
        "unit_sales": "sales"
    },
    inplace=True
)

sales_store_day.to_parquet(
    rf"{OUTPUT_PATH}\sales_store_day.parquet",
    index=False
)

print(sales_store_day.shape)
print("sales_store_day listo")

Creando sales_store_day...
(83606, 3)
sales_store_day listo


In [4]:
#SALES_FAMILY_DAY.PARQUET
import pandas as pd
from collections import defaultdict

print("Procesando sales_family_day por chunks...")

RAW_PATH = r"C:\Users\34642\Desktop\FLOWMAP ANALYTICS\segundo proyecto\POWER BI\data\raw"
OUTPUT_PATH = r"C:\Users\34642\Desktop\FLOWMAP ANALYTICS\segundo proyecto\POWER BI\data_processed"

items = pd.read_csv(f"{RAW_PATH}\\items.csv")

# mapa item -> family
item_family = dict(zip(items["item_nbr"], items["family"]))

agg = defaultdict(float)

chunksize = 200_000

for i, chunk in enumerate(
    pd.read_csv(f"{RAW_PATH}\\train.csv", chunksize=chunksize)
):

    print(f"Chunk {i+1}")

    chunk["family"] = chunk["item_nbr"].map(item_family)

    g = chunk.groupby(
        ["date", "family"]
    )["unit_sales"].sum()

    for k, v in g.items():
        agg[k] += v

# convertir a dataframe
df_family = pd.DataFrame(
    [(k[0], k[1], v) for k, v in agg.items()],
    columns=["date", "family", "sales"]
)

df_family.to_parquet(
    f"{OUTPUT_PATH}\\sales_family_day.parquet",
    index=False
)

print("OK sales_family_day")

Procesando sales_family_day por chunks...
Chunk 1
Chunk 2
Chunk 3
Chunk 4
Chunk 5
Chunk 6
Chunk 7
Chunk 8
Chunk 9
Chunk 10
Chunk 11
Chunk 12
Chunk 13
Chunk 14
Chunk 15
Chunk 16
Chunk 17
Chunk 18
Chunk 19
Chunk 20
Chunk 21
Chunk 22
Chunk 23
Chunk 24
Chunk 25
Chunk 26
Chunk 27
Chunk 28
Chunk 29
Chunk 30
Chunk 31
Chunk 32
Chunk 33
Chunk 34
Chunk 35
Chunk 36
Chunk 37
Chunk 38
Chunk 39
Chunk 40
Chunk 41
Chunk 42
Chunk 43
Chunk 44
Chunk 45
Chunk 46
Chunk 47
Chunk 48
Chunk 49
Chunk 50
Chunk 51
Chunk 52
Chunk 53
Chunk 54
Chunk 55
Chunk 56
Chunk 57
Chunk 58
Chunk 59
Chunk 60
Chunk 61
Chunk 62
Chunk 63
Chunk 64
Chunk 65
Chunk 66
Chunk 67
Chunk 68
Chunk 69
Chunk 70
Chunk 71
Chunk 72
Chunk 73
Chunk 74
Chunk 75
Chunk 76
Chunk 77
Chunk 78
Chunk 79
Chunk 80
Chunk 81
Chunk 82
Chunk 83
Chunk 84
Chunk 85
Chunk 86
Chunk 87
Chunk 88
Chunk 89
Chunk 90
Chunk 91
Chunk 92
Chunk 93
Chunk 94
Chunk 95
Chunk 96
Chunk 97
Chunk 98
Chunk 99
Chunk 100
Chunk 101
Chunk 102
Chunk 103
Chunk 104
Chunk 105
Chunk 106
Chunk 

C:\Users\34642\AppData\Local\Temp\ipykernel_13212\3013393625.py:19: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(


Chunk 109
Chunk 110
Chunk 111
Chunk 112
Chunk 113
Chunk 114
Chunk 115
Chunk 116
Chunk 117
Chunk 118
Chunk 119
Chunk 120
Chunk 121
Chunk 122
Chunk 123
Chunk 124
Chunk 125
Chunk 126
Chunk 127
Chunk 128
Chunk 129
Chunk 130
Chunk 131
Chunk 132
Chunk 133
Chunk 134
Chunk 135
Chunk 136
Chunk 137
Chunk 138
Chunk 139
Chunk 140
Chunk 141
Chunk 142
Chunk 143
Chunk 144
Chunk 145
Chunk 146
Chunk 147
Chunk 148
Chunk 149
Chunk 150
Chunk 151
Chunk 152
Chunk 153
Chunk 154
Chunk 155
Chunk 156
Chunk 157
Chunk 158
Chunk 159
Chunk 160
Chunk 161
Chunk 162
Chunk 163
Chunk 164
Chunk 165
Chunk 166
Chunk 167
Chunk 168
Chunk 169
Chunk 170
Chunk 171
Chunk 172
Chunk 173
Chunk 174
Chunk 175
Chunk 176
Chunk 177
Chunk 178
Chunk 179
Chunk 180
Chunk 181
Chunk 182
Chunk 183
Chunk 184
Chunk 185
Chunk 186
Chunk 187
Chunk 188
Chunk 189
Chunk 190
Chunk 191
Chunk 192
Chunk 193
Chunk 194
Chunk 195
Chunk 196
Chunk 197
Chunk 198
Chunk 199
Chunk 200
Chunk 201
Chunk 202
Chunk 203
Chunk 204
Chunk 205
Chunk 206
Chunk 207
Chunk 208


In [6]:
#SALES_CITY_DAY.PARQUET
import pandas as pd
from collections import defaultdict

print("Procesando sales_city_day por chunks...")

RAW_PATH = r"C:\Users\34642\Desktop\FLOWMAP ANALYTICS\segundo proyecto\POWER BI\data\raw"
OUTPUT_PATH = r"C:\Users\34642\Desktop\FLOWMAP ANALYTICS\segundo proyecto\POWER BI\data_processed"

stores = pd.read_csv(f"{RAW_PATH}\\stores.csv")

store_city = dict(zip(stores["store_nbr"], stores["city"]))

agg = defaultdict(float)

chunksize = 200_000

for i, chunk in enumerate(
    pd.read_csv(f"{RAW_PATH}\\train.csv", chunksize=chunksize)
):

    print(f"Chunk {i+1}")

    chunk["city"] = chunk["store_nbr"].map(store_city)

    g = chunk.groupby(
        ["date", "city"]
    )["unit_sales"].sum()

    for k, v in g.items():
        agg[k] += v

df_city = pd.DataFrame(
    [(k[0], k[1], v) for k, v in agg.items()],
    columns=["date", "city", "sales"]
)

df_city.to_parquet(
    f"{OUTPUT_PATH}\\sales_city_day.parquet",
    index=False
)

print("OK sales_city_day")

Procesando sales_city_day por chunks...
Chunk 1
Chunk 2
Chunk 3
Chunk 4
Chunk 5
Chunk 6
Chunk 7
Chunk 8
Chunk 9
Chunk 10
Chunk 11
Chunk 12
Chunk 13
Chunk 14
Chunk 15
Chunk 16
Chunk 17
Chunk 18
Chunk 19
Chunk 20
Chunk 21
Chunk 22
Chunk 23
Chunk 24
Chunk 25
Chunk 26
Chunk 27
Chunk 28
Chunk 29
Chunk 30
Chunk 31
Chunk 32
Chunk 33
Chunk 34
Chunk 35
Chunk 36
Chunk 37
Chunk 38
Chunk 39
Chunk 40
Chunk 41
Chunk 42
Chunk 43
Chunk 44
Chunk 45
Chunk 46
Chunk 47
Chunk 48
Chunk 49
Chunk 50
Chunk 51
Chunk 52
Chunk 53
Chunk 54
Chunk 55
Chunk 56
Chunk 57
Chunk 58
Chunk 59
Chunk 60
Chunk 61
Chunk 62
Chunk 63
Chunk 64
Chunk 65
Chunk 66
Chunk 67
Chunk 68
Chunk 69
Chunk 70
Chunk 71
Chunk 72
Chunk 73
Chunk 74
Chunk 75
Chunk 76
Chunk 77
Chunk 78
Chunk 79
Chunk 80
Chunk 81
Chunk 82
Chunk 83
Chunk 84
Chunk 85
Chunk 86
Chunk 87
Chunk 88
Chunk 89
Chunk 90
Chunk 91
Chunk 92
Chunk 93
Chunk 94
Chunk 95
Chunk 96
Chunk 97
Chunk 98
Chunk 99
Chunk 100
Chunk 101
Chunk 102
Chunk 103
Chunk 104
Chunk 105
Chunk 106
Chunk 10

C:\Users\34642\AppData\Local\Temp\ipykernel_13212\4217225980.py:18: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(


Chunk 109
Chunk 110
Chunk 111
Chunk 112
Chunk 113
Chunk 114
Chunk 115
Chunk 116
Chunk 117
Chunk 118
Chunk 119
Chunk 120
Chunk 121
Chunk 122
Chunk 123
Chunk 124
Chunk 125
Chunk 126
Chunk 127
Chunk 128
Chunk 129
Chunk 130
Chunk 131
Chunk 132
Chunk 133
Chunk 134
Chunk 135
Chunk 136
Chunk 137
Chunk 138
Chunk 139
Chunk 140
Chunk 141
Chunk 142
Chunk 143
Chunk 144
Chunk 145
Chunk 146
Chunk 147
Chunk 148
Chunk 149
Chunk 150
Chunk 151
Chunk 152
Chunk 153
Chunk 154
Chunk 155
Chunk 156
Chunk 157
Chunk 158
Chunk 159
Chunk 160
Chunk 161
Chunk 162
Chunk 163
Chunk 164
Chunk 165
Chunk 166
Chunk 167
Chunk 168
Chunk 169
Chunk 170
Chunk 171
Chunk 172
Chunk 173
Chunk 174
Chunk 175
Chunk 176
Chunk 177
Chunk 178
Chunk 179
Chunk 180
Chunk 181
Chunk 182
Chunk 183
Chunk 184
Chunk 185
Chunk 186
Chunk 187
Chunk 188
Chunk 189
Chunk 190
Chunk 191
Chunk 192
Chunk 193
Chunk 194
Chunk 195
Chunk 196
Chunk 197
Chunk 198
Chunk 199
Chunk 200
Chunk 201
Chunk 202
Chunk 203
Chunk 204
Chunk 205
Chunk 206
Chunk 207
Chunk 208


In [7]:
#PRODUCT_SUMMARY.PARQUET
import pandas as pd
from collections import defaultdict

print("Procesando product_summary por chunks...")

RAW_PATH = r"C:\Users\34642\Desktop\FLOWMAP ANALYTICS\segundo proyecto\POWER BI\data\raw"
OUTPUT_PATH = r"C:\Users\34642\Desktop\FLOWMAP ANALYTICS\segundo proyecto\POWER BI\data_processed"

items = pd.read_csv(f"{RAW_PATH}\\items.csv")

# Mapas ligeros (MUY importante para performance)
item_family = dict(zip(items["item_nbr"], items["family"]))
item_class = dict(zip(items["item_nbr"], items["class"]))
item_perishable = dict(zip(items["item_nbr"], items["perishable"]))

agg = defaultdict(float)

chunksize = 200_000

for i, chunk in enumerate(
    pd.read_csv(f"{RAW_PATH}\\train.csv", chunksize=chunksize)
):

    print(f"Chunk {i+1}")

    # enrich sin merge (MUY IMPORTANTE)
    chunk["family"] = chunk["item_nbr"].map(item_family)
    chunk["class"] = chunk["item_nbr"].map(item_class)
    chunk["perishable"] = chunk["item_nbr"].map(item_perishable)

    g = chunk.groupby(
        ["item_nbr", "family", "class", "perishable"]
    )["unit_sales"].sum()

    for k, v in g.items():
        agg[k] += v

# convertir a dataframe final
df_product = pd.DataFrame(
    [
        (k[0], k[1], k[2], k[3], v)
        for k, v in agg.items()
    ],
    columns=[
        "item_nbr",
        "family",
        "class",
        "perishable",
        "sales"
    ]
)

# ordenar tipo Pareto (muy útil para Power BI)
df_product = df_product.sort_values(
    "sales",
    ascending=False
)

df_product.to_parquet(
    rf"{OUTPUT_PATH}\product_summary.parquet",
    index=False
)

print("OK product_summary")
print(df_product.shape)

Procesando product_summary por chunks...
Chunk 1
Chunk 2
Chunk 3
Chunk 4
Chunk 5
Chunk 6
Chunk 7
Chunk 8
Chunk 9
Chunk 10
Chunk 11
Chunk 12
Chunk 13
Chunk 14
Chunk 15
Chunk 16
Chunk 17
Chunk 18
Chunk 19
Chunk 20
Chunk 21
Chunk 22
Chunk 23
Chunk 24
Chunk 25
Chunk 26
Chunk 27
Chunk 28
Chunk 29
Chunk 30
Chunk 31
Chunk 32
Chunk 33
Chunk 34
Chunk 35
Chunk 36
Chunk 37
Chunk 38
Chunk 39
Chunk 40
Chunk 41
Chunk 42
Chunk 43
Chunk 44
Chunk 45
Chunk 46
Chunk 47
Chunk 48
Chunk 49
Chunk 50
Chunk 51
Chunk 52
Chunk 53
Chunk 54
Chunk 55
Chunk 56
Chunk 57
Chunk 58
Chunk 59
Chunk 60
Chunk 61
Chunk 62
Chunk 63
Chunk 64
Chunk 65
Chunk 66
Chunk 67
Chunk 68
Chunk 69
Chunk 70
Chunk 71
Chunk 72
Chunk 73
Chunk 74
Chunk 75
Chunk 76
Chunk 77
Chunk 78
Chunk 79
Chunk 80
Chunk 81
Chunk 82
Chunk 83
Chunk 84
Chunk 85
Chunk 86
Chunk 87
Chunk 88
Chunk 89
Chunk 90
Chunk 91
Chunk 92
Chunk 93
Chunk 94
Chunk 95
Chunk 96
Chunk 97
Chunk 98
Chunk 99
Chunk 100
Chunk 101
Chunk 102
Chunk 103
Chunk 104
Chunk 105
Chunk 106
Chunk 1

C:\Users\34642\AppData\Local\Temp\ipykernel_13212\611275185.py:21: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(


Chunk 109
Chunk 110
Chunk 111
Chunk 112
Chunk 113
Chunk 114
Chunk 115
Chunk 116
Chunk 117
Chunk 118
Chunk 119
Chunk 120
Chunk 121
Chunk 122
Chunk 123
Chunk 124
Chunk 125
Chunk 126
Chunk 127
Chunk 128
Chunk 129
Chunk 130
Chunk 131
Chunk 132
Chunk 133
Chunk 134
Chunk 135
Chunk 136
Chunk 137
Chunk 138
Chunk 139
Chunk 140
Chunk 141
Chunk 142
Chunk 143
Chunk 144
Chunk 145
Chunk 146
Chunk 147
Chunk 148
Chunk 149
Chunk 150
Chunk 151
Chunk 152
Chunk 153
Chunk 154
Chunk 155
Chunk 156
Chunk 157
Chunk 158
Chunk 159
Chunk 160
Chunk 161
Chunk 162
Chunk 163
Chunk 164
Chunk 165
Chunk 166
Chunk 167
Chunk 168
Chunk 169
Chunk 170
Chunk 171
Chunk 172
Chunk 173
Chunk 174
Chunk 175
Chunk 176
Chunk 177
Chunk 178
Chunk 179
Chunk 180
Chunk 181
Chunk 182
Chunk 183
Chunk 184
Chunk 185
Chunk 186
Chunk 187
Chunk 188
Chunk 189
Chunk 190
Chunk 191
Chunk 192
Chunk 193
Chunk 194
Chunk 195
Chunk 196
Chunk 197
Chunk 198
Chunk 199
Chunk 200
Chunk 201
Chunk 202
Chunk 203
Chunk 204
Chunk 205
Chunk 206
Chunk 207
Chunk 208


In [8]:
#GUARDAAR DIMENSIONES
items.to_parquet(
    rf"{OUTPUT_PATH}\items.parquet",
    index=False
)

stores.to_parquet(
    rf"{OUTPUT_PATH}\stores.parquet",
    index=False
)

print("Dimensiones guardadas.")

Dimensiones guardadas.
